# Construct pydantic model from text input

In [ ]:
from pydantic_ai import Agent

# this is quite fast model that also has tool use
agent = Agent(model="openrouter:arcee-ai/trinity-mini:free")

result = await agent.run("Give me an IT employee working in sweden, shortly")
result

AgentRunResult(output="\n\nHere's a concise description of anIT employee working in Sweden:\n\n*   **Location:** Primarily based in major tech hubs like Stockholm, Gothenburg, or Malmö.\n*   **Role:** Works in software development, IT operations (DevOps, SysAdmin), cybersecurity, data science, or related fields.\n*   **Work Culture:** Operates in a **flat organizational structure** with strong emphasis on **collaboration, autonomy, and work-life balance**. Open communication and consensus-building are valued.\n*   **Skills:** Requires strong technical skills (programming languages, cloud platforms, tools) and often includes **English proficiency** (as the primary language for tech discussions, even in Swedish companies).\n*   **Benefits:** Access to **generous parental leave, extensive vacation time, and a strong focus on employee well-being**.\n*   **Environment:** Works for a mix of **international tech giants, innovative startups, and established Swedish companies** leveraging cutti

In [2]:
print(result.output)



Here's a concise description of anIT employee working in Sweden:

*   **Location:** Primarily based in major tech hubs like Stockholm, Gothenburg, or Malmö.
*   **Role:** Works in software development, IT operations (DevOps, SysAdmin), cybersecurity, data science, or related fields.
*   **Work Culture:** Operates in a **flat organizational structure** with strong emphasis on **collaboration, autonomy, and work-life balance**. Open communication and consensus-building are valued.
*   **Skills:** Requires strong technical skills (programming languages, cloud platforms, tools) and often includes **English proficiency** (as the primary language for tech discussions, even in Swedish companies).
*   **Benefits:** Access to **generous parental leave, extensive vacation time, and a strong focus on employee well-being**.
*   **Environment:** Works for a mix of **international tech giants, innovative startups, and established Swedish companies** leveraging cutting-edge technology.


In [3]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(description="salary should be between 30k and 50k, the more senior the position, the higher salary", gt=30_000, lt=50_000)
    position: str


result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Anna Svensson', age=35, salary=45000, position='Senior Software Engineer'))

In [4]:
result.output

EmployeeModel(name='Anna Svensson', age=35, salary=45000, position='Senior Software Engineer')

In [5]:
result.output.name, result.output.age, result.output.salary

('Anna Svensson', 35, 45000)

In [6]:
result.output.model_dump()

{'name': 'Anna Svensson',
 'age': 35,
 'salary': 45000,
 'position': 'Senior Software Engineer'}

In [7]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[EmployeeModel],
)
result

AgentRunResult(output=[EmployeeModel(name='John Doe', age=35, salary=45000, position='Data Scientist'), EmployeeModel(name='Jane Smith', age=40, salary=48000, position='Machine Learning Engineer'), EmployeeModel(name='Alex Johnson', age=30, salary=42000, position='AI Researcher'), EmployeeModel(name='Mike Brown', age=38, salary=38000, position='Data Engineer'), EmployeeModel(name='Sarah Davis', age=45, salary=45000, position='Senior Data Engineer'), EmployeeModel(name='Tom Wilson', age=32, salary=35000, position='AI Engineer'), EmployeeModel(name='Lisa Miller', age=37, salary=47000, position='Machine Learning Engineer'), EmployeeModel(name='Kevin Taylor', age=29, salary=32000, position='Data Analyst'), EmployeeModel(name='Emily Clark', age=41, salary=49000, position='Principal Data Scientist'), EmployeeModel(name='David Evans', age=36, salary=44000, position='AI Architect')])

In [8]:
result.output

[EmployeeModel(name='John Doe', age=35, salary=45000, position='Data Scientist'),
 EmployeeModel(name='Jane Smith', age=40, salary=48000, position='Machine Learning Engineer'),
 EmployeeModel(name='Alex Johnson', age=30, salary=42000, position='AI Researcher'),
 EmployeeModel(name='Mike Brown', age=38, salary=38000, position='Data Engineer'),
 EmployeeModel(name='Sarah Davis', age=45, salary=45000, position='Senior Data Engineer'),
 EmployeeModel(name='Tom Wilson', age=32, salary=35000, position='AI Engineer'),
 EmployeeModel(name='Lisa Miller', age=37, salary=47000, position='Machine Learning Engineer'),
 EmployeeModel(name='Kevin Taylor', age=29, salary=32000, position='Data Analyst'),
 EmployeeModel(name='Emily Clark', age=41, salary=49000, position='Principal Data Scientist'),
 EmployeeModel(name='David Evans', age=36, salary=44000, position='AI Architect')]

## CV model - a more complex and nested model

In [34]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int


class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    educations: list[EducationModel]


result = await agent.run(
    "Create 5 fake persons that is applying for a data engineering job. They should have 2-5 previous job experiences",
    output_type=list[CvModel],
)



In [35]:
result.output[-1]

CvModel(name='David Wilson', age=31, experiences=[ExperienceModel(title='Data Engineer', company='Analytics Inc.', description='Designed and deployed a data lake on AWS, including data ingestion, storage, and processing using Spark.', start_year=2021, end_year=2023), ExperienceModel(title='Senior Data Engineer', company='DataFlow', description='Created a data pipeline for a logistics company, optimizing route planning using geospatial data.', start_year=2021, end_year=2023), ExperienceModel(title='Data Engineer', company='Insight Analytics', description='Developed a real-time dashboard for monitoring data pipeline performance, using Grafana and Prometheus.', start_year=2019, end_year=2021), ExperienceModel(title='Data Warehouse Engineer', company='DataWorks', description='Built a data warehouse for a media company, improving data retrieval speed by 70%.', start_year=2017, end_year=2019)], educations=[EducationModel(title='MSc Computer Science', education_area='Computer Science', school

In [36]:
result.output[-1].name

'David Wilson'

In [37]:
result.output[-1].experiences[0].title, result.output[-1].experiences[1].title

('Data Engineer', 'Senior Data Engineer')

## (optional) Postprocessing - load into duckdb and unnesting

This part is optional, but a way to unnest the data and store it could be to use dlt to load the data into duckdb, followed by joining and unnesting.

Other approach could be to store into nosql such as mongodb.

In [41]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="cv_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging",
)

info = pipeline.run(
    data=[result.output], loader_file_format="jsonl", table_name="cv_entries"
)

print(info)


Pipeline cv_duckdb load step completed in 0.03 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:////Users/aigineer/Documents/teaching repos/llmops_course/04_pydanticai_structured_output/cv.duckdb location to store data
Load package 1773436078.379661 is LOADED and contains no failed jobs


In [42]:
import duckdb 

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc;").df()
    cv_entries = conn.sql("FROM staging.cv_entries").df()
    educations = conn.sql("FROM staging.cv_entries__educations").df()
    experiences = conn.sql("FROM staging.cv_entries__experiences").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__educations,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experiences,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [43]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,John Doe,28,1773436078.379661,vS8Gy+hhoLs9pA
1,Sarah Johnson,32,1773436078.379661,gbzW4ohR6qvlXQ
2,Michael Chen,35,1773436078.379661,Z4AfDMtlCoGqKg
3,Emily Davis,29,1773436078.379661,anAyU43ilkCt5Q
4,David Wilson,31,1773436078.379661,d2+NrKbbsPSnZQ


In [44]:
educations

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,BSc Computer Science,Computer Science,Stanford University,Bachelor of Science in Computer Science,2015,2018,vS8Gy+hhoLs9pA,0,gaQBlBBq/O3ijQ
1,MSc Data Science,Data Science,MIT,Master of Data Science,2019,2021,gbzW4ohR6qvlXQ,0,7j4F2ktzJjjvsA
2,BEng Computer Science,Computer Science,UC Berkeley,Bachelor of Engineering in Computer Science,2012,2016,Z4AfDMtlCoGqKg,0,MVZiFbTkOatZUg
3,BSc Data Science,Data Science,University of Toronto,Bachelor of Science in Data Science,2016,2020,anAyU43ilkCt5Q,0,i7WCLBUHPELFGg
4,MSc Computer Science,Computer Science,Georgia Tech,Master of Science in Computer Science,2020,2022,d2+NrKbbsPSnZQ,0,93t8pNGEw1Ttgw


In [45]:
experiences

,title,company,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,Data Engineer,TechCorp,Developed ETL pipelines using Apache Spark and...,2020,2022,vS8Gy+hhoLs9pA,0,mMiJOP31GGinbQ
1,Senior Data Engineer,DataSolutions,Designed and implemented data warehousing solu...,2018,2020,vS8Gy+hhoLs9pA,1,oLRlfOgCCdd1zw
2,Data Engineering Lead,CloudTech,Built and maintained data pipelines for real-t...,2021,2023,gbzW4ohR6qvlXQ,0,MT9y+lbcINc2pw
3,Data Engineer,DataFlow,Optimized database queries and schema design f...,2019,2021,gbzW4ohR6qvlXQ,1,OvKwInnpQuYtxg
4,ML Engineer,Analytics Inc.,Developed machine learning models for predicti...,2017,2019,gbzW4ohR6qvlXQ,2,fg+mJypXLVNbXQ
5,Data Engineer,DataWorks,Created and managed data pipelines for a finte...,2021,2023,Z4AfDMtlCoGqKg,0,Mf1tp51DNn6vXw
6,Senior Data Engineer,CodeHub,Designed a data lake architecture on AWS S3 an...,2019,2021,Z4AfDMtlCoGqKg,1,Ie3jxXmS2XnUtQ
7,Data Engineer,DataStream,Developed real-time data processing systems us...,2017,2019,Z4AfDMtlCoGqKg,2,9Gi8CST3zapCcQ
8,Data Warehouse Architect,Insight Analytics,"Built a data warehouse for a retail client, im...",2015,2017,Z4AfDMtlCoGqKg,3,VLr50zuzOdXpsA
9,Data Quality Engineer,DataSolutions,Implemented data validation and quality checks...,2020,2022,anAyU43ilkCt5Q,0,br1gsASGE+N+FA


In [46]:
duckdb.sql("""
    SELECT 
        cv.name, 
        cv.age, 
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
    

""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,John Doe,28,TechCorp,Developed ETL pipelines using Apache Spark and...,2020,2022,BSc Computer Science,Computer Science,Stanford University,2015,2018
1,John Doe,28,DataSolutions,Designed and implemented data warehousing solu...,2018,2020,BSc Computer Science,Computer Science,Stanford University,2015,2018
2,Sarah Johnson,32,CloudTech,Built and maintained data pipelines for real-t...,2021,2023,MSc Data Science,Data Science,MIT,2019,2021
3,Sarah Johnson,32,DataFlow,Optimized database queries and schema design f...,2019,2021,MSc Data Science,Data Science,MIT,2019,2021
4,Sarah Johnson,32,Analytics Inc.,Developed machine learning models for predicti...,2017,2019,MSc Data Science,Data Science,MIT,2019,2021
5,Michael Chen,35,DataWorks,Created and managed data pipelines for a finte...,2021,2023,BEng Computer Science,Computer Science,UC Berkeley,2012,2016
6,Michael Chen,35,CodeHub,Designed a data lake architecture on AWS S3 an...,2019,2021,BEng Computer Science,Computer Science,UC Berkeley,2012,2016
7,Michael Chen,35,DataStream,Developed real-time data processing systems us...,2017,2019,BEng Computer Science,Computer Science,UC Berkeley,2012,2016
8,Michael Chen,35,Insight Analytics,"Built a data warehouse for a retail client, im...",2015,2017,BEng Computer Science,Computer Science,UC Berkeley,2012,2016
9,Emily Davis,29,DataSolutions,Implemented data validation and quality checks...,2020,2022,BSc Data Science,Data Science,University of Toronto,2016,2020
